# Notebook 5: Sex/Gender Content Analysis

Identify articles containing sex/gender analysis in their abstracts and classify them into three categories:
1. **Instrumental (INS):** Gender as predictor of financial outcomes (e.g., female CEOs and firm performance)
2. **Catalogues of Sex Differences (CSD):** Documenting behavioral differences
3. **Gender Mechanisms (GM):** Investigating structural/systemic gender dynamics

**Paper findings:** Only 0.78% of articles (433/55,210) contained sex/gender analysis.
Split: 51% INS, 32% CSD, 17% GM

**Output:** `data/processed/articles_gender_content.parquet`

In [1]:
import pandas as pd
import sys
sys.path.insert(0, ".")
from utils import has_gender_content, classify_gender_article, GENDER_PATTERN

# Load articles with topics (1988-2020, with abstracts)
articles_df = pd.read_parquet("data/processed/articles_with_topics.parquet")
print(f"Articles with abstracts (1988-2020): {len(articles_df):,}")

Articles with abstracts (1988-2020): 47,724


## 1. Keyword Search for Sex/Gender Content

In [2]:
# Flag articles with sex/gender content
articles_df["has_gender_content"] = articles_df["abstract"].apply(has_gender_content)

n_gender = articles_df["has_gender_content"].sum()
pct_gender = n_gender / len(articles_df) * 100

print(f"Articles with sex/gender content: {n_gender:,} ({pct_gender:.2f}%)")
print(f"Paper reference: 433 articles (0.78%)")

# Show which keywords matched most often
import re
from collections import Counter

keyword_counts = Counter()
for abstract in articles_df[articles_df["has_gender_content"]]["abstract"]:
    matches = GENDER_PATTERN.findall(abstract)
    for m in matches:
        keyword_counts[m.lower()] += 1

print(f"\nTop matched keywords:")
for keyword, count in keyword_counts.most_common(15):
    print(f"  {keyword}: {count}")

Articles with sex/gender content: 496 (1.04%)
Paper reference: 433 articles (0.78%)

Top matched keywords:
  women: 468
  gender: 468
  female: 354
  male: 164
  board diversity: 56
  sex: 33
  masculin: 21
  woman: 10
  wage gap: 8
  girl: 1
  feminin: 1
  glass ceiling: 1


## 2. Classify Gender Articles into Categories

Using a heuristic keyword classifier. For a full replication, manually review all flagged articles.

In [3]:
# Classify gender articles
gender_articles = articles_df[articles_df["has_gender_content"]].copy()
gender_articles["gender_category"] = gender_articles["abstract"].apply(classify_gender_article)

# Report distribution
print("Gender content category distribution:")
cat_counts = gender_articles["gender_category"].value_counts()
cat_pcts = gender_articles["gender_category"].value_counts(normalize=True) * 100

for cat in ["instrumental", "sex_differences_catalogue", "gender_mechanisms"]:
    count = cat_counts.get(cat, 0)
    pct = cat_pcts.get(cat, 0)
    print(f"  {cat:30s}: {count:4d} ({pct:.1f}%)")

print(f"\nPaper reference:")
print(f"  Instrumental:         51%")
print(f"  Sex differences:      32%")
print(f"  Gender mechanisms:    17%")

# Add category to full dataframe
articles_df["gender_category"] = None
articles_df.loc[articles_df["has_gender_content"], "gender_category"] = (
    gender_articles["gender_category"].values
)

Gender content category distribution:
  instrumental                  :  337 (67.9%)
  sex_differences_catalogue     :   68 (13.7%)
  gender_mechanisms             :   91 (18.3%)

Paper reference:
  Instrumental:         51%
  Sex differences:      32%
  Gender mechanisms:    17%


In [4]:
# Show sample articles from each category for manual review
for cat in ["instrumental", "sex_differences_catalogue", "gender_mechanisms"]:
    subset = gender_articles[gender_articles["gender_category"] == cat]
    print(f"\n{'='*80}")
    print(f"SAMPLE: {cat.upper()} (n={len(subset)})")
    print(f"{'='*80}")
    for _, row in subset.head(3).iterrows():
        print(f"\nTitle: {row['title']}")
        print(f"Year: {row['publication_year']}")
        abstract = row["abstract"][:300] + "..." if len(str(row["abstract"])) > 300 else row["abstract"]
        print(f"Abstract: {abstract}")
        print("-" * 40)


SAMPLE: INSTRUMENTAL (n=337)

Title: The Armchair Economist: Economics and Everyday Life.
Year: 1994
Abstract: Why does popcorn cost so much at the movies? When does it make sense not to recycle? Why are laws against polygamy detrimental to women? Steven E. Landsburg examines everything from taxes, unemployment and illiteracy to the mating game, the death penalty and environmentalism to solve the puzzling qu...
----------------------------------------

Title: Women's Liberation as a Financial Innovation
Year: 2019
Abstract: ABSTRACT In one of the greatest extensions of property rights in human history, common law countries began giving rights to married women in the 1850s. Before this “women's liberation,” the doctrine of coverture strongly incentivized parents of daughters to hold real estate, rather than financial as...
----------------------------------------

Title: Financial Attention
Year: 2015
Abstract: This paper investigates financial attention using novel panel data on daily

## 3. Save Results

In [5]:
# Save
articles_df.to_parquet("data/processed/articles_gender_content.parquet", index=False)
print(f"Saved: data/processed/articles_gender_content.parquet ({len(articles_df):,} rows)")
print(f"  with has_gender_content=True: {articles_df['has_gender_content'].sum():,}")

# Also export gender articles to CSV for manual review
gender_articles[["work_id", "title", "publication_year", "journal_name", "abstract", "gender_category"]].to_csv(
    "data/processed/gender_articles_for_review.csv", index=False
)
print(f"  Exported gender articles for manual review: data/processed/gender_articles_for_review.csv")

Saved: data/processed/articles_gender_content.parquet (47,724 rows)
  with has_gender_content=True: 496
  Exported gender articles for manual review: data/processed/gender_articles_for_review.csv
